# Analyse territoriale des flux d’azote

# Objectif

Ce projet vise à simuler les surplus et déficits d’azote entre exploitations agricoles afin d’explorer les complémentarités cultures-élevage et la circularité des nutriments à l’échelle territoriale.

L’objectif est d’identifier les échanges potentiels de nutriments entre les exploitations présentant un surplus d’azote et celles présentant un déficit d’azote.

In [217]:
import pandas as pd

df = pd.read_csv("../data/farms.csv")

df

,Farm,Type,Area_ha,Cattle,Nitrogen_Produced,Nitrogen_Needed,Distance_km,Latitude,Longitude
0,Farm_A,Livestock,50,80,120,30,5,48.11,-1.67
1,Farm_B,Crop,70,0,10,100,12,48.13,-1.60
2,Farm_C,Mixed,40,30,60,50,7,48.10,-1.72
3,Farm_D,Crop,90,0,15,130,15,48.15,-1.65
4,Farm_E,Livestock,60,100,150,40,9,48.08,-1.70
5,Farm_F,Crop,10,50,70,10,8,48.20,-1.50
6,Farm_G,Crop,90,0,15,130,15,48.18,-1.40


Calculer le bilan azoté

In [218]:

df["Nitrogen_Balance"] = (
    df["Nitrogen_Produced"]
    -
    df["Nitrogen_Needed"]
)

df

,Farm,Type,Area_ha,Cattle,Nitrogen_Produced,Nitrogen_Needed,Distance_km,Latitude,Longitude,Nitrogen_Balance
0,Farm_A,Livestock,50,80,120,30,5,48.11,-1.67,90
1,Farm_B,Crop,70,0,10,100,12,48.13,-1.60,-90
2,Farm_C,Mixed,40,30,60,50,7,48.10,-1.72,10
3,Farm_D,Crop,90,0,15,130,15,48.15,-1.65,-115
4,Farm_E,Livestock,60,100,150,40,9,48.08,-1.70,110
5,Farm_F,Crop,10,50,70,10,8,48.20,-1.50,60
6,Farm_G,Crop,90,0,15,130,15,48.18,-1.40,-115


Séparer surplus et déficit

In [219]:
surplus_farms = df[df["Nitrogen_Balance"] > 0]

deficit_farms = df[df["Nitrogen_Balance"] < 0]

print("SURPLUS")
print(surplus_farms[["Farm", "Nitrogen_Balance"]])

print("\nDEFICIT")
print(deficit_farms[["Farm", "Nitrogen_Balance"]])

SURPLUS
     Farm  Nitrogen_Balance
0  Farm_A                90
2  Farm_C                10
4  Farm_E               110
5  Farm_F                60

DEFICIT
     Farm  Nitrogen_Balance
1  Farm_B               -90
3  Farm_D              -115
6  Farm_G              -115


Ajout coût transport

In [220]:

df["Transport_Cost"] = (
    df["Distance_km"] * 2
)

Créer une logique scientifique

Hypothèse :

La ferme excédentaire la plus proche alimente la ferme déficitaire.

Mais :

sans dépasser son surplus,
sans dépasser le déficit du receveur.

Créer les échanges manuels intelligents

In [221]:
transfers = [
    ("Farm_A", "Farm_B", 90),
    ("Farm_E", "Farm_D", 110),
    ("Farm_F", "Farm_G", 60)
]

transfer_df = pd.DataFrame(
    transfers,
    columns=[
        "Source",
        "Destination",
        "Nitrogen_Transferred"
    ]
)

transfer_df

,Source,Destination,Nitrogen_Transferred
0,Farm_A,Farm_B,90
1,Farm_E,Farm_D,110
2,Farm_F,Farm_G,60


Ajouter les coordonnées pour la carte

In [222]:
farm_coords = {
    row["Farm"]: [row["Latitude"], row["Longitude"]]
    for _, row in df.iterrows()
}

Créer la carte

In [223]:
import folium

map_farms = folium.Map(
    location=[48.11, -1.67],
    zoom_start=9
)

Ajouter les fermes

In [224]:
for _, row in df.iterrows():

    color = "green" if row["Nitrogen_Balance"] > 0 else "red"

    folium.CircleMarker(
        location=[
            row["Latitude"],
            row["Longitude"]
        ],
        radius=8,
        color=color,
        fill=True,
        fill_color=color,
        popup=f"{row['Farm']} | Balance: {row['Nitrogen_Balance']}"
    ).add_to(map_farms)

Tracer les flux

In [225]:
for source, dest, qty in transfers:

    folium.PolyLine(
        locations=[
            farm_coords[source],
            farm_coords[dest]
        ],
        tooltip=f"{qty} units transferred"
    ).add_to(map_farms)

map_farms

Automatiser les échanges

In [226]:
import pulp

Créer le problème d’optimisation

In [227]:
problem = pulp.LpProblem(
    "Territorial_Nitrogen_Optimization",
    pulp.LpMinimize
)

Créer les variables automatiques

In [228]:
flows = {}

for s_idx, s_row in surplus_farms.iterrows():

    for d_idx, d_row in deficit_farms.iterrows():

        var_name = f"{s_row['Farm']}_to_{d_row['Farm']}"

        flows[(s_row["Farm"], d_row["Farm"])] = pulp.LpVariable(
            var_name,
            lowBound=0
        )

Fonction objectif

In [229]:
objective = []

for s_idx, s_row in surplus_farms.iterrows():

    for d_idx, d_row in deficit_farms.iterrows():

        distance = abs(
            s_row["Distance_km"]
            -
            d_row["Distance_km"]
        )

        flow = flows[(s_row["Farm"], d_row["Farm"])]

        objective.append(distance * flow)

problem += pulp.lpSum(objective)


In [230]:
print("Total optimization cost:", pulp.value(problem.objective))

Total optimization cost: None


Contraintes de surplus

In [231]:
for s_idx, s_row in surplus_farms.iterrows():

    outgoing = []

    for d_idx, d_row in deficit_farms.iterrows():

        outgoing.append(
            flows[(s_row["Farm"], d_row["Farm"])]
        )

    problem += (
        pulp.lpSum(outgoing)
        <=
        s_row["Nitrogen_Balance"]
    )

Contraintes de déficit

In [232]:
for d_idx, d_row in deficit_farms.iterrows():

    incoming = []

    for s_idx, s_row in surplus_farms.iterrows():

        incoming.append(
            flows[(s_row["Farm"], d_row["Farm"])]
        )

    problem += (
        pulp.lpSum(incoming)
        <=
        abs(d_row["Nitrogen_Balance"])
    )

Résoudre le problème

In [233]:
problem.solve()

1

In [234]:
print("Status:", pulp.LpStatus[problem.status])

print("\nOptimal Nitrogen Transfers:\n")

for key, variable in flows.items():

    if variable.varValue > 0:

        print(
            f"{key[0]} -> {key[1]} : {variable.varValue:.2f} units"
        )

Status: Optimal

Optimal Nitrogen Transfers:



Voir les échanges optimaux

In [235]:
for key, variable in flows.items():

    if variable.varValue > 0:

        print(
            f"{key[0]} -> {key[1]} : {variable.varValue}"
        )

# Conclusion générale de la recherche exploratoire

Cette étude exploratoire a permis de développer un prototype simplifié de modélisation territoriale des flux d’azote entre exploitations agricoles afin d’évaluer le potentiel des complémentarités cultures-élevage dans une logique de circularité des nutriments.

La base de données utilisée comprenait sept exploitations agricoles caractérisées selon leur type de production, leur surface, leur cheptel, leurs besoins en azote et leur production potentielle d’azote :

| Exploitation | Type      | Surface (ha) | Cheptel | Azote produit | Azote nécessaire |
| ------------ | --------- | ------------ | ------- | ------------- | ---------------- |
| Farm_A       | Livestock | 50           | 80      | 120           | 30               |
| Farm_B       | Crop      | 70           | 0       | 10            | 100              |
| Farm_C       | Mixed     | 40           | 30      | 60            | 50               |
| Farm_D       | Crop      | 90           | 0       | 15            | 130              |
| Farm_E       | Livestock | 60           | 100     | 150           | 40               |
| Farm_F       | Crop      | 10           | 50      | 70            | 10               |
| Farm_G       | Crop      | 90           | 0       | 15            | 130              |

Le calcul du bilan azoté a permis de mettre en évidence des situations contrastées au sein du territoire :

| Exploitation | Bilan azoté |
| ------------ | ----------- |
| Farm_A       | +90         |
| Farm_B       | -90         |
| Farm_C       | +10         |
| Farm_D       | -115        |
| Farm_E       | +110        |
| Farm_F       | +60         |
| Farm_G       | -115        |

Ces résultats montrent l’existence de forts déséquilibres territoriaux. Certaines exploitations d’élevage présentent des excédents importants en azote, tandis que plusieurs exploitations de cultures dépendent fortement d’apports extérieurs en fertilisants.

L’analyse spatiale réalisée grâce aux coordonnées géographiques des exploitations a permis d’introduire une dimension territoriale essentielle. La cartographie des exploitations a mis en évidence des opportunités potentielles de redistribution locale des nutriments entre fermes excédentaires et déficitaires.

Une première logique d’échanges territoriaux a ainsi été simulée :

| Source | Destination | Azote transféré |
| ------ | ----------- | --------------- |
| Farm_A | Farm_B      | 90              |
| Farm_E | Farm_D      | 110             |
| Farm_F | Farm_G      | 60              |

Ces transferts permettent :

* d’équilibrer totalement le déficit de Farm_B ;
* de réduire fortement le déficit de Farm_D ;
* de diminuer partiellement le déficit de Farm_G.

L’étude a ensuite introduit une logique simplifiée d’optimisation territoriale à l’aide de PuLP. Le modèle cherchait à minimiser les coûts liés aux distances de transport tout en respectant plusieurs contraintes :

* une exploitation ne peut pas transférer plus que son surplus ;
* une exploitation déficitaire ne peut pas recevoir plus que ses besoins ;
* les échanges doivent rester cohérents spatialement.

Cette approche démontre qu’une meilleure organisation territoriale des flux de biomasse pourrait contribuer à :

* améliorer la circularité des nutriments ;
* réduire les excédents responsables de pollutions azotées ;
* limiter l’utilisation d’engrais minéraux ;
* renforcer les interactions entre productions animales et végétales ;
* favoriser une agriculture plus durable et territorialisée.

Même simplifié, ce prototype met déjà en évidence l’intérêt des approches de modélisation spatiale et d’optimisation pour analyser les systèmes agricoles complexes. Il constitue une première base méthodologique cohérente pour explorer les enjeux de complémentarité cultures-élevage à l’échelle territoriale.

Enfin, plusieurs perspectives d’amélioration pourraient être envisagées dans un travail de recherche plus avancé :

* intégration de distances géographiques réelles ;
* prise en compte des coûts économiques du transport ;
* intégration des émissions de gaz à effet de serre ;
* ajout de scénarios de méthanisation ;
* prise en compte des coproduits agroalimentaires ;
* développement d’une optimisation multicritère environnementale et économique.

Cette étude exploratoire illustre ainsi le potentiel des outils numériques et de la modélisation pour accompagner la transition agroécologique des territoires agricoles.
